# 01 - Nền tảng CNN

Notebook này chuẩn bị nền tảng toán học và trực giác cho các kiến trúc CNN trong Assignment 05. Mục tiêu là hiểu CNN như một chuỗi hàm được ghép lại, chưa xây dựng hay huấn luyện mô hình trên dataset thật.


In [1]:
from pathlib import Path
import os
import sys
import random

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    current = current.resolve()
    if current.is_file():
        current = current.parent
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "datasets").is_dir():
            return candidate
    raise RuntimeError("A05 project root was not found.")

PROJECT_ROOT = find_project_root()

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {__import__('matplotlib').__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Pillow version: {Image.__version__}")
print(f"TensorFlow devices: {tf.config.list_physical_devices()}")
print(f"Project root detected: {PROJECT_ROOT.name}")

Python executable: C:\Users\anhca\anaconda3\envs\tf312\python.exe
Python version: 3.12.14
NumPy version: 2.5.3
Matplotlib version: 3.11.2
TensorFlow version: 2.21.0
Keras version: 3.15.1
Pillow version: 12.3.0
TensorFlow devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Project root detected: A05


## 1. CNN như một hợp của các hàm

Một mạng nơ-ron có thể được hiểu như một hàm lớn được tạo bằng cách ghép nhiều hàm nhỏ:

\[
f(x) = f_5(f_4(f_3(f_2(f_1(x)))))
\]

Nghĩa là dữ liệu đầu vào đi qua nhiều phép biến đổi liên tiếp. Trong CNN, các phép biến đổi thường gồm convolution, activation, pooling, và các lớp phân loại ở cuối.


In [2]:
def f1(x):
    return x + 2


def f2(x):
    return 3 * x


def f3(x):
    return x - 1

x = 4
composition_result = f3(f2(f1(x)))
print({"x": x, "f3(f2(f1(x)))": composition_result})

{'x': 4, 'f3(f2(f1(x)))': 17}


## 2. Convolution

Phép convolution 2D dùng một kernel/filter nhỏ trượt trên ảnh hoặc ma trận đầu vào để tạo feature map.

\[
Y(i,j) = \sum_m \sum_n X(i+m, j+n)K(m,n)
\]

Trong đó:

- \(X\) là input.
- \(K\) là kernel/filter.
- \(Y\) là feature map đầu ra.

Convolution giúp mô hình học các mẫu cục bộ như cạnh, vùng sáng/tối, texture, hoặc cấu trúc hình học nhỏ.


In [3]:
def conv2d_valid(input_matrix, kernel):
    """Compute a valid 2D cross-correlation, the operation commonly used by CNN libraries."""
    input_matrix = np.asarray(input_matrix, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    out_height = input_matrix.shape[0] - kernel.shape[0] + 1
    out_width = input_matrix.shape[1] - kernel.shape[1] + 1
    output = np.zeros((out_height, out_width), dtype=float)

    for i in range(out_height):
        for j in range(out_width):
            patch = input_matrix[i:i + kernel.shape[0], j:j + kernel.shape[1]]
            output[i, j] = np.sum(patch * kernel)
    return output

input_matrix = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16],
])

kernel = np.array([
    [1, 0],
    [0, -1],
])

manual_position_00 = 1 * 1 + 2 * 0 + 5 * 0 + 6 * (-1)
conv_output = conv2d_valid(input_matrix, kernel)

print("Input matrix:")
print(input_matrix)
print("Kernel:")
print(kernel)
print(f"Manual output at (0, 0): {manual_position_00}")
print("Complete output feature map:")
print(conv_output)

Input matrix:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]
Kernel:
[[ 1  0]
 [ 0 -1]]
Manual output at (0, 0): -5
Complete output feature map:
[[-5. -5. -5.]
 [-5. -5. -5.]
 [-5. -5. -5.]]


Kernel/filter là tập trọng số nhỏ được học trong quá trình huấn luyện. Khi kernel trượt qua ảnh, nó phản ứng mạnh với một kiểu mẫu cục bộ nào đó, ví dụ cạnh, vùng sáng/tối, hoặc texture. Nhiều kernel khác nhau tạo ra nhiều feature map khác nhau.


## 3. ReLU

ReLU là hàm kích hoạt:

\[
\mathrm{ReLU}(z) = \max(0, z)
\]

Giá trị âm trở thành 0, giá trị dương được giữ lại. ReLU đưa tính phi tuyến vào mạng. Nếu nhiều tầng chỉ là biến đổi tuyến tính, toàn bộ mạng vẫn tương đương một biến đổi tuyến tính lớn; activation giúp CNN biểu diễn quan hệ phức tạp hơn.


In [4]:
def relu(values):
    return np.maximum(values, 0)

relu_input = conv_output
relu_output = relu(relu_input)

print("Input to ReLU:")
print(relu_input)
print("Output after ReLU:")
print(relu_output)

Input to ReLU:
[[-5. -5. -5.]
 [-5. -5. -5.]
 [-5. -5. -5.]]
Output after ReLU:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


## 4. Pooling

MaxPooling lấy giá trị lớn nhất trong mỗi cửa sổ cục bộ. Ví dụ với cửa sổ `2x2` và stride `2`, mỗi vùng `2x2` được thay bằng giá trị lớn nhất của vùng đó.

Pooling làm giảm kích thước không gian của feature map, giảm chi phí tính toán, và giúp biểu diễn bớt nhạy với dịch chuyển nhỏ trong ảnh.


In [5]:
def max_pool2d(feature_map, pool_size=(2, 2), stride=2):
    feature_map = np.asarray(feature_map, dtype=float)
    pool_height, pool_width = pool_size
    out_height = (feature_map.shape[0] - pool_height) // stride + 1
    out_width = (feature_map.shape[1] - pool_width) // stride + 1
    output = np.zeros((out_height, out_width), dtype=float)

    for i in range(out_height):
        for j in range(out_width):
            row = i * stride
            col = j * stride
            window = feature_map[row:row + pool_height, col:col + pool_width]
            output[i, j] = np.max(window)
    return output

pool_input = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [0, 2, 8, 1],
    [3, 1, 4, 7],
])

pool_output = max_pool2d(pool_input, pool_size=(2, 2), stride=2)

print("Input feature map:")
print(pool_input)
print("Pooling window: 2x2")
print("Stride: 2")
print("Resulting output feature map:")
print(pool_output)

Input feature map:
[[1 3 2 4]
 [5 6 1 2]
 [0 2 8 1]
 [3 1 4 7]]
Pooling window: 2x2
Stride: 2
Resulting output feature map:
[[6. 4.]
 [3. 8.]]


## 5. Flatten, Dense, và Softmax

Sau khi trích xuất đặc trưng không gian, mạng cần chuyển đặc trưng thành biểu diễn phù hợp cho phân loại.

- `Flatten` biến feature map nhiều chiều thành vector dài.
- `Dense` học kết hợp các đặc trưng đã trích xuất.
- `Softmax` biến logits thành xác suất trên các lớp:

\[
p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}
\]


In [6]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)

flattened = pool_output.flatten()
dense_weights = np.array([
    [0.2, -0.1, 0.3],
    [0.0, 0.4, -0.2],
    [0.1, 0.2, 0.1],
    [-0.3, 0.1, 0.2],
])
dense_bias = np.array([0.1, -0.2, 0.0])
class_scores = flattened @ dense_weights + dense_bias
class_probabilities = softmax(class_scores)

print("Flattened representation:")
print(flattened)
print("Class scores:")
print(class_scores)
print("Softmax probabilities:")
print(np.round(class_probabilities, 4))
print(f"Probability sum: {class_probabilities.sum():.4f}")

Flattened representation:
[6. 4. 3. 8.]
Class scores:
[-0.8  2.2  2.9]
Softmax probabilities:
[0.0163 0.3264 0.6573]
Probability sum: 1.0000


## 6. Kiểm tra khái niệm với Keras, chưa xây dựng kiến trúc

Phần này chỉ kiểm tra rằng các phép toán khái niệm tương thích với cách thư viện hiện thực các lớp cơ bản. Notebook này chưa tạo kiến trúc cho Assignment 05 và chưa huấn luyện model.


In [7]:
keras_relu = keras.layers.ReLU()(tf.constant(relu_input, dtype=tf.float32)).numpy()
keras_softmax = keras.activations.softmax(tf.constant(class_scores, dtype=tf.float32)).numpy()

print("NumPy ReLU equals Keras ReLU:", np.allclose(relu_output, keras_relu))
print("NumPy Softmax close to Keras Softmax:", np.allclose(class_probabilities, keras_softmax))

NumPy ReLU equals Keras ReLU: True
NumPy Softmax close to Keras Softmax: True


## 7. Tham số học được và siêu tham số

**Tham số học được** là các giá trị được cập nhật từ dữ liệu huấn luyện, ví dụ:

- trọng số kernel của convolution,
- bias của convolution,
- trọng số của Dense layer.

**Siêu tham số** là các lựa chọn do người thiết kế giao thức quyết định trước hoặc chọn bằng validation, ví dụ:

- learning rate,
- batch size,
- số filter,
- kernel size,
- số block,
- dropout,
- `max_epochs`,
- `EarlyStopping` patience.


## 8. Bốn loại quyết định tham số trong Assignment 05

| Loại quyết định | Ý nghĩa | Ví dụ |
| --- | --- | --- |
| data-determined | Giá trị suy ra trực tiếp từ dữ liệu | số kênh vào, số lớp, shape ảnh EuroSAT |
| architecture-determined | Giá trị bắt buộc bởi cấu trúc mô hình | projection shortcut khi số channel khác nhau trong residual block |
| experimentally selected | Giá trị chọn bằng validation evidence | learning rate, batch size, image resolution |
| operational bound | Giới hạn để thí nghiệm chạy được trên CPU/thời gian có sẵn | `max_epochs`, kích thước tuning subset |

Không dùng các lý do như "commonly used" hoặc "popular choice" để chọn siêu tham số quan trọng.


## 9. Chiến lược giải mã ảnh

Không được giả định định dạng ảnh từ phần mở rộng tên file. Oxford-IIIT Pet có bốn file tên `.jpg` nhưng nội dung thật là PNG. Vì vậy pipeline sau này phải dùng bộ giải mã dựa trên nội dung, ví dụ `tf.io.decode_image` hoặc Pillow `Image.open`, rồi convert sang RGB trong bộ nhớ. File ảnh gốc không được sửa.


In [8]:
from PIL import Image

known_png_content_jpg = [
    "datasets/oxford_pets/images/Abyssinian_5.jpg",
    "datasets/oxford_pets/images/Egyptian_Mau_14.jpg",
    "datasets/oxford_pets/images/Egyptian_Mau_156.jpg",
    "datasets/oxford_pets/images/Egyptian_Mau_186.jpg",
]

format_check = []
for relative_path in known_png_content_jpg:
    image_path = PROJECT_ROOT / relative_path
    with Image.open(image_path) as image:
        rgb = image.convert("RGB")
        format_check.append((Path(relative_path).name, image.format, rgb.mode, rgb.size))

for row in format_check:
    print(row)

('Abyssinian_5.jpg', 'PNG', 'RGB', (200, 150))
('Egyptian_Mau_14.jpg', 'PNG', 'RGB', (582, 800))
('Egyptian_Mau_156.jpg', 'PNG', 'RGB', (400, 265))
('Egyptian_Mau_186.jpg', 'PNG', 'RGB', (183, 275))


## 10. Chính sách mẫu chính thức của Oxford-IIIT Pet

Trong các thí nghiệm phân loại Oxford, chỉ các mẫu được tham chiếu bởi:

- `annotations/trainval.txt`,
- `annotations/test.txt`

mới được dùng. Không glob toàn bộ thư mục `images/` để tạo dataset, vì có 41 ảnh raw không thuộc tập mẫu chính thức.


In [9]:
def parse_oxford_annotation(path):
    records = []
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            text = line.strip()
            if not text or text.startswith("#"):
                continue
            image_id, class_id, species, breed_id = text.split()[:4]
            records.append({
                "image_id": image_id,
                "class_id": int(class_id),
                "species": int(species),
                "breed_id": int(breed_id),
            })
    return records

trainval_records = parse_oxford_annotation(PROJECT_ROOT / "datasets/oxford_pets/annotations/trainval.txt")
test_records = parse_oxford_annotation(PROJECT_ROOT / "datasets/oxford_pets/annotations/test.txt")
trainval_ids = {record["image_id"] for record in trainval_records}
test_ids = {record["image_id"] for record in test_records}

print({
    "trainval": len(trainval_records),
    "test": len(test_records),
    "total": len(trainval_records) + len(test_records),
    "overlap": len(trainval_ids & test_ids),
    "classes": len({record["class_id"] for record in trainval_records + test_records}),
})

{'trainval': 3680, 'test': 3669, 'total': 7349, 'overlap': 0, 'classes': 37}


## 11. Không dùng test set để chọn siêu tham số

Validation dùng để chọn mô hình và siêu tham số. Test set chỉ dùng sau cùng để đánh giá cấu hình đã được chọn. Nếu dùng test set để chọn learning rate, batch size, dropout, hoặc biến thể kiến trúc, kết quả test không còn là đánh giá độc lập nữa.


## 12. Kết nối đến phát triển kiến trúc CNN

Các kiến trúc sau sẽ được triển khai trong `A05/models/architectures.py` ở bước sau, chưa triển khai trong notebook này:

- **Basic CNN**: baseline convolution đơn giản.
- **AlexNet-inspired**: tăng chiều sâu và năng lực biểu diễn.
- **VGG-inspired**: lặp lại các block convolution kernel nhỏ.
- **ResNet-inspired**: dùng residual learning với `y = F(x) + x`.


## 13. Tóm tắt

Luồng khái niệm của CNN là:

`Image/Input -> local feature extraction -> nonlinear transformation -> spatial reduction -> learned representation -> class scores -> class probabilities`

Notebook này chỉ thiết lập trực giác và quy tắc thực nghiệm. Các kết quả mô hình thật phải đến từ các notebook thí nghiệm sau.
